# Dunnhumby M5-C 최소 귀속 게이트 (개발구간)

- 고정 범위: seed 42·44, 300 epoch, 개발평가 DAY 684~690
- 신규 학습 4회: M4-C 단독 2회 + M5-C CLV 공동순열 대조 2회
- M1·M3·관측 CLV M5-C는 기존 완료 결과만 재사용
- 마지막 주 test·holdout·조기종료·epoch 선택 없음


In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys

if not os.path.ismount('/content/drive'):
    try:
        drive.mount('/content/drive')
    except (ValueError, OSError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패입니다. 런타임을 재시작한 뒤 이 셀부터 다시 실행하세요. 학습은 시작되지 않았습니다.') from exc

ROOT = Path('/content/drive/MyDrive/논문/data')
for seed in (42, 44):
    old = ROOT / f'results_v3_dunnhumby_clv_m5_m3_m4_user_centered_s{seed}_v1'
    matches = list(old.glob('clv-m5-m3-m4-user-centered-dev-v1_*.json'))
    if len(matches) != 1:
        raise RuntimeError(f'기존 observed M5-C seed{seed} 완료 JSON이 정확히 하나 필요합니다: {old}. 신규 학습 전에 중단했습니다.')
print('기존 seed42·44 M5-C 결과 확인 완료 — 신규 4개 arm만 학습합니다.')


In [ ]:
SOURCE_COMMIT = 'c2f514a742c8a67cde27fa2598f9201877114e0b'
REPO = Path('/content/clv-m5c-attribution-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
import clv_m5_m3_m4_user_centered_attribution as screen
screen.observed.self_test()
print(json.dumps({
    'source_commit': SOURCE_COMMIT,
    'split': 'historical_development_days_684_690',
    'seeds': list(screen.SEEDS),
    'epochs': screen.FIXED_EPOCH,
    'new_fits': [screen.M4_C_MODEL_ID, screen.M5_C_SHUFFLED_MODEL_ID],
    'fit_count': 4,
    'reused': [screen.m3.M1_MODEL_ID, screen.m3.ARM_VALUE_ACTIVITY, screen.observed.MODEL_ID],
}, ensure_ascii=False, indent=2))


In [ ]:
# 중단돼도 같은 셀을 다시 실행하면 완료 arm은 재사용하고 미완료 arm은 epoch 체크포인트에서 재개합니다.
result = screen.run()


In [ ]:
from IPython.display import display
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10', 'vndcg@10',
    'price_purchase_amount_weighted_hit@20', 'vndcg@20',
    'price_purchase_amount_weighted_hit@50', 'vndcg@50',
]
display(result['absolute'].query('epoch == 300')[['model_id', 'seed', 'epoch', *metrics]])
display(result['comparison'].query('epoch == 300 and metric in @metrics'))
display(result['factorial'].query('metric in @metrics'))
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

zip_path = Path('/content/clv_m5_m3_m4_user_centered_attribution_seed42_44_results.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, Path(path).name)
files.download(str(zip_path))
